# LAB 06 — Delta MERGE & Change Data Capture (CDC)
### Retail Customer 360 · Databricks Solutions Architect Interview Lab

**Platform:** Databricks Free Edition (Serverless) · **Language:** Python / PySpark / Delta Lake / Spark SQL  
**Source:** Existing public GitHub CSVs under `normanmartinez2/retail_customer360/data/crm/`  
**Prerequisites:** Complete LAB 02–05. In particular, `silver.customers_scd2`, `silver.orders_clean`, and `silver.payments_clean` must exist.  
**Scope:** 680 CDC source rows (180 customers, 300 orders, 200 payments). **All explanations and code comments are in English.**

> **Safety contract:** This lab NEVER overwrites LAB 03's `silver.customers_scd2` or LAB 05's `silver.orders_clean` / `silver.payments_clean`. It creates separately named LAB 06 targets and reads existing tables as immutable input snapshots.

> **Reality check:** The files are *synthetic CDC exports*, not a live connector or Debezium feed. The notebook demonstrates SCD2 MERGE, partial updates, replay deduplication, source-time ordering, soft deletes, and current-state MERGE. See the limitations near the end before calling it production-ready.

## Architecture & outcomes

```text
GitHub: customers_cdc / orders_cdc / payments_cdc
                       |
          Managed Unity Catalog Volume
                       |
                BRONZE CDC (raw)
                       |
        Parse / validate / deduplicate IDs
                       |
         +-------------+--------------+
         |             |              |
         v             v              v
   CRM patch stream  Orders changes  Payment changes
         |             |              |
   Event-time SCD2   Delta MERGE    Delta MERGE
         |             |              |
   customers_scd2_  orders_current_ payments_current_
       lab06             lab06             lab06
         |             |              |
         +-------------+--------------+
                       |
          QA, replay test, UC governance
```

**Business goal:** Preserve changes to customer attributes, apply order/payment status updates without double-counting transactions, and explain why event-time and ingestion-time are different.

**Distinct responsibilities**

- **Bronze CDC:** append-free snapshot of exactly the GitHub file bytes plus source provenance, never cleaned in place.
- **CDC validation:** typed timestamps, required keys, business rules, and replay resolution.
- **Customers:** track *every meaningful event-time change within this controlled file* as SCD2, including soft-deletion tombstones; apply them atomically in a staged Delta MERGE.
- **Orders / Payments:** maintain current-state snapshots using conditional Delta MERGE; a payment status change is **not** a new payment transaction.
- **Audit:** demonstrate rejects, duplicate event IDs, source ordering, reconciliation, and idempotency.

## STEP 01 — Configure the catalog, table names, and prerequisites

Use fully qualified Unity Catalog names. Stop early if LAB 03 or LAB 05 has not been completed.

In [ ]:
# Set a deterministic timezone for source event timestamps with UTC ('Z') suffix.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable
import hashlib
import csv
import os
import urllib.request

CATALOG = "retail_customer360"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
REPO = "https://github.com/normanmartinez2/retail_customer360"
RAW_BASE = "https://raw.githubusercontent.com/normanmartinez2/retail_customer360/refs/heads/main/data/crm"
BATCH_ID = "LAB06_CDC_V1"

BASE_CUSTOMERS = f"{SILVER}.customers_scd2"
BASE_ORDERS = f"{SILVER}.orders_clean"
BASE_PAYMENTS = f"{SILVER}.payments_clean"
CUSTOMERS_TARGET = f"{SILVER}.customers_scd2_lab06"
ORDERS_TARGET = f"{SILVER}.orders_current_lab06"
PAYMENTS_TARGET = f"{SILVER}.payments_current_lab06"
CUSTOMERS_VIEW = f"{SILVER}.customers_current_lab06"
ORDERS_VIEW = f"{SILVER}.orders_active_lab06"
PAYMENTS_VIEW = f"{SILVER}.payments_active_lab06"

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("USE SCHEMA silver")

for required in (BASE_CUSTOMERS, BASE_ORDERS, BASE_PAYMENTS):
    assert spark.catalog.tableExists(required), f"Missing prerequisite {required}. Complete LAB 03/05 first."

print("Existing source dimensions confirmed.")
print("Safe LAB 06 targets:", CUSTOMERS_TARGET, ORDERS_TARGET, PAYMENTS_TARGET)

## STEP 02 — Declare the real GitHub CDC contracts

All three CSVs are in **`data/crm/`**, even though their logical business domains differ. Exact SHA-256 hashes make the lab reproducible if the `main` branch later changes.

In [ ]:
# IMPORTANT: These hashes describe the exact original CSV bytes created for the lab.
# All fields remain STRING in Bronze so malformed source values are not hidden.
CDC = {
    "customers": {
        "filename": "customers_cdc_lab06_v1.csv", "expected_rows": 180,
        "sha256": "13c6b4155d0cf6e3d56fd5b86071550c3bf3d6561686513ce11c4355d2555d30",
        "columns": ["event_id", "operation", "customer_id", "event_timestamp", "source_sequence",
                    "first_name", "last_name", "email", "phone", "date_of_birth", "country", "city",
                    "customer_segment", "marketing_consent", "created_at", "updated_at", "source_system",
                    "notes", "ingestion_timestamp"]
    },
    "orders": {
        "filename": "orders_cdc_lab06_v1.csv", "expected_rows": 300,
        "sha256": "70a280eb8793fc1918d832fbe349adc763ac41fdf295a4181dbd64af7a4c7d9e",
        "columns": ["event_id", "operation", "order_id", "event_timestamp", "source_sequence",
                    "new_order_status", "new_total_amount_eur", "ingestion_timestamp",
                    "source_system", "notes"]
    },
    "payments": {
        "filename": "payments_cdc_lab06_v1.csv", "expected_rows": 200,
        "sha256": "fb150a063540766a528787decb5a345c4252f207e2cf7d08d1a24ad2b5c1c02c",
        "columns": ["event_id", "operation", "payment_id", "order_id", "event_timestamp",
                    "source_sequence", "new_payment_status", "new_amount_eur",
                    "ingestion_timestamp", "source_system", "notes"]
    }
}

assert sum(s["expected_rows"] for s in CDC.values()) == 680
for domain, spec in CDC.items():
    spec["url"] = f"{RAW_BASE}/{spec['filename']}"
    spec["bronze"] = f"{BRONZE}.{domain}_cdc_raw"
    print(domain, "→", spec["url"])

## STEP 03 — Land the CDC files in a governed Volume

The Volume preserves source bytes independently of Delta tables. If Serverless blocks outbound HTTPS, manually upload the original file to the displayed `/Volumes/...` path and rerun the cell.

In [ ]:
# Create an independent LAB 06 landing zone; do not modify the LAB 04 Volume.
CDC_VOLUME = f"{BRONZE}.cdc_source_files"
CDC_VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/cdc_source_files"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CDC_VOLUME} COMMENT 'Immutable CDC lab file landing zone'")

for domain, spec in CDC.items():
    spec["path"] = f"{CDC_VOLUME_PATH}/{spec['filename']}"
print("CDC landing volume:", CDC_VOLUME_PATH)

In [ ]:
# File verification is deliberately independent of Spark's CSV parser.
def sha256_file(path, chunk_size=1024 * 1024):
    """Return a streaming SHA-256 digest of the unmodified source file."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def ensure_landed(spec):
    """Reuse an exact matching source file; otherwise download it from GitHub."""
    path = spec["path"]
    if os.path.exists(path):
        if sha256_file(path) != spec["sha256"]:
            raise ValueError(f"Existing Volume file has an unexpected checksum: {path}")
        return "EXISTING_FILE_VERIFIED"
    req = urllib.request.Request(spec["url"], headers={"User-Agent": "RetailCustomer360-LAB06"})
    try:
        with urllib.request.urlopen(req, timeout=45) as response, open(path, "wb") as out:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
        if sha256_file(path) != spec["sha256"]:
            raise ValueError("The downloaded file differs from the pinned source version")
    except Exception as exc:
        if os.path.exists(path):
            os.remove(path)
        raise RuntimeError(
            f"Unable to download {spec['filename']}. Upload the ORIGINAL CSV manually "
            f"from {spec['url']} to {path}, then rerun. Cause: {exc}"
        ) from exc
    return "DOWNLOADED_AND_VERIFIED"

for domain, spec in CDC.items():
    print(domain, ensure_landed(spec))

## STEP 04 — Verify original headers, count rows with Spark, and create immutable Bronze CDC tables

Use `FAILFAST` for broken CSV syntax; keep business-quality problems as strings for the next step. **Do not append the same source file again on replay.**

In [ ]:
# Validate headers before any Delta writes.
for domain, spec in CDC.items():
    with open(spec["path"], encoding="utf-8-sig", newline="") as handle:
        header = next(csv.reader(handle))
    assert header == spec["columns"], f"Unexpected {domain} columns: {header}"
    assert sha256_file(spec["path"]) == spec["sha256"]
    print(f"PASS: {domain} header and SHA-256")

In [ ]:
# Create each managed Delta table exactly once and verify it on notebook reruns.
CDC_META = ["source_file", "source_file_url", "source_file_sha256",
            "ingestion_batch_id", "bronze_ingestion_timestamp"]

for domain, spec in CDC.items():
    schema = StructType([StructField(c, StringType(), True) for c in spec["columns"]])
    raw = (spark.read.format("csv")
        .option("header", "true")
        .option("mode", "FAILFAST")
        .option("enforceSchema", "false")
        .option("encoding", "UTF-8")
        .schema(schema)
        .load(spec["path"]))
    actual = raw.count()
    assert actual == spec["expected_rows"], f"{domain}: expected {spec['expected_rows']}, got {actual}"
    if not spark.catalog.tableExists(spec["bronze"]):
        tagged = (raw.withColumn("source_file", F.lit(spec["filename"]))
            .withColumn("source_file_url", F.lit(spec["url"]))
            .withColumn("source_file_sha256", F.lit(spec["sha256"]))
            .withColumn("ingestion_batch_id", F.lit(BATCH_ID))
            .withColumn("bronze_ingestion_timestamp", F.current_timestamp()))
        tagged.write.format("delta").mode("errorifexists").saveAsTable(spec["bronze"])
        print("CREATED", spec["bronze"])
    else:
        print("REUSED", spec["bronze"])
    persisted = spark.table(spec["bronze"])
    assert persisted.count() == spec["expected_rows"]
    assert persisted.columns == spec["columns"] + CDC_META
    assert persisted.select("source_file_sha256").distinct().collect()[0][0] == spec["sha256"]

print("PASS: 680 raw CDC rows are present, without replay appends.")

## STEP 05 — Explain the CDC event contract before processing

- `event_id`: source event identity; duplicates are **replays**, not new business operations.
- `operation`: INSERT / UPDATE / DELETE (supported operations differ by domain).
- `event_timestamp`: business-event time, used for chronology.
- `source_sequence`: extra source ordering clue; **not** a universal global offset in this lab.
- `ingestion_timestamp`: arrival time, which may be much later than event time.
- Empty fields in CRM `UPDATE` are **partial-patch omissions**, not instructions to erase values. A true field-clear operation needs explicit presence flags or before/after payloads; this feed does not contain them.
- `DELETE`: soft deletion (tombstone), not destructive Delta `DELETE`.
- `new_amount_eur` / `new_total_amount_eur`: optionally supplied replacements; blank means keep previous amount.

**Important source detail:** intentionally replayed `event_id`s may have a different *notes* annotation, but otherwise identical business payloads. Verify that before resolving duplicates.

## STEP 06 — Normalize, parse, and deduplicate replayed events

Deduplicate by `event_id`, *not* by customer ID or order ID. Multiple distinct changes for one customer must remain available for SCD2.

In [ ]:
# Transform only CDC working DataFrames; Bronze remains unchanged.
def normalize_cdc(df):
    """Trim source text, convert blanks to NULL, and create typed CDC control fields."""
    for name, kind in df.dtypes:
        if kind == "string" and name not in CDC_META:
            cleaned = F.trim(F.col(name))
            df = df.withColumn(name, F.when(cleaned == "", F.lit(None)).otherwise(cleaned))
    return (df
        .withColumn("operation", F.upper("operation"))
        .withColumn("event_at", F.expr("try_cast(event_timestamp AS TIMESTAMP)"))
        .withColumn("arrived_at", F.expr("try_cast(ingestion_timestamp AS TIMESTAMP)"))
        .withColumn("event_sequence", F.expr("try_cast(source_sequence AS INT)")))


def deduplicate_event_ids(df, domain):
    """Resolve only true event-ID replays; annotation-only differences are permitted."""
    # The business payload excludes free-text 'notes' (a replay annotation in this fixture).
    business_fields = [c for c in CDC[domain]["columns"] if c != "notes"]
    # A second event with the same ID but different business payload is a source conflict.
    payload = df.withColumn("__payload_sha", F.sha2(F.to_json(
        F.struct(*[F.col(c) for c in business_fields]),
        options={"ignoreNullFields": "false"}), 256))
    conflicts = (payload.groupBy("event_id")
        .agg(F.countDistinct("__payload_sha").alias("payload_count"))
        .filter("payload_count > 1"))
    assert conflicts.count() == 0, f"Conflicting event-id payloads in {domain}; investigate first"
    # Ranking stabilizes the annotation retained for a replayed event.
    w = Window.partitionBy("event_id").orderBy(F.col("notes").asc_nulls_last())
    ranked = payload.withColumn("__replay_rank", F.row_number().over(w))
    canonical = ranked.filter(F.col("__replay_rank") == 1).drop("__replay_rank", "__payload_sha")
    replays = ranked.filter(F.col("__replay_rank") > 1).drop("__replay_rank", "__payload_sha")
    return canonical, replays

canonical = {}
replayed = {}
for domain in CDC:
    incoming = normalize_cdc(spark.table(CDC[domain]["bronze"]))
    canonical[domain], replayed[domain] = deduplicate_event_ids(incoming, domain)
    print(f"{domain}: source={incoming.count()}, unique events={canonical[domain].count()}, "
          f"replayed IDs={replayed[domain].count()}")

# These counts are measured source-fixture contracts (180/300/200 rows).
assert canonical["customers"].count() == 174
assert canonical["orders"].count() == 288
assert canonical["payments"].count() == 192
assert sum(replayed[x].count() for x in CDC) == 26

## STEP 07 — Apply CDC-specific quality rules and record rejects

The same event schema is **not** expected across different source systems. Validate operation types and PATCH fields separately.

In [ ]:
# Build an array of validation error codes without erasing original values.
def error_array(rules):
    """Return an array of every violated quality-rule code."""
    return F.array_compact(F.array(*[
        F.when(F.coalesce(cond, F.lit(False)), F.lit(rule_id))
        for rule_id, cond in rules
    ]))

validated = {}
valid_events = {}
invalid_events = {}

for domain, df in canonical.items():
    key = {"customers": "customer_id", "orders": "order_id", "payments": "payment_id"}[domain]
    permitted = (["INSERT", "UPDATE", "DELETE"] if domain == "customers"
                 else ["UPDATE", "DELETE"] if domain == "orders" else ["UPDATE"])
    rules = [
        ("CDC001_MISSING_EVENT_ID", F.col("event_id").isNull()),
        ("CDC002_INVALID_OPERATION", F.col("operation").isNull() | ~F.col("operation").isin(permitted)),
        ("CDC003_MISSING_BUSINESS_KEY", F.col(key).isNull()),
        ("CDC004_INVALID_EVENT_TIMESTAMP", F.col("event_at").isNull()),
        ("CDC005_INVALID_ARRIVAL_TIMESTAMP", F.col("arrived_at").isNull()),
        ("CDC006_INVALID_SEQUENCE", F.col("event_sequence").isNull() | (F.col("event_sequence") < 0)),
    ]
    if domain == "customers":
        rules += [
            ("CDC007_INSERT_MISSING_NAME", (F.col("operation") == "INSERT") &
             (F.col("first_name").isNull() | F.col("last_name").isNull())),
            ("CDC008_INSERT_INVALID_EMAIL", (F.col("operation") == "INSERT") &
             (F.col("email").isNotNull()) & ~F.regexp_like(F.col("email"), r"^[^@\s]+@[^@\s]+\.[^@\s]+$")),
        ]
    if domain == "orders":
        df = (df.withColumn("new_order_status", F.upper("new_order_status"))
            .withColumn("new_amount_parsed", F.expr("try_cast(new_total_amount_eur AS DECIMAL(18,2))")))
        rules += [
            ("CDC009_INVALID_ORDER_STATUS", (F.col("operation") == "UPDATE") &
             (F.col("new_order_status").isNull() | ~F.col("new_order_status").isin(
                 "COMPLETED", "CANCELLED", "RETURNED", "PENDING"))),
            ("CDC010_INVALID_ORDER_AMOUNT", F.col("new_total_amount_eur").isNotNull() &
             (F.col("new_amount_parsed").isNull() | (F.col("new_amount_parsed") < 0))),
        ]
    if domain == "payments":
        df = (df.withColumn("new_payment_status", F.upper("new_payment_status"))
            .withColumn("new_amount_parsed", F.expr("try_cast(new_amount_eur AS DECIMAL(18,2))")))
        rules += [
            ("CDC011_INVALID_PAYMENT_STATUS", (F.col("operation") == "UPDATE") &
             (F.col("new_payment_status").isNull() | ~F.col("new_payment_status").isin(
                 "CAPTURED", "DECLINED", "PENDING", "SUCCEEDED"))),
            ("CDC012_INVALID_PAYMENT_AMOUNT", F.col("new_amount_eur").isNotNull() &
             F.col("new_amount_parsed").isNull()),
        ]
    flagged = (df.withColumn("dq_errors", error_array(rules))
        .withColumn("dq_status", F.when(F.size("dq_errors") == 0, "VALID").otherwise("INVALID")))
    validated[domain] = flagged
    valid_events[domain] = flagged.filter("dq_status = 'VALID'")
    invalid_events[domain] = flagged.filter("dq_status = 'INVALID'")
    print(domain, "valid", valid_events[domain].count(), "invalid", invalid_events[domain].count())

# Invalid rows are retained in working DataFrames; they are never silently dropped.

## STEP 08 — Inspect event time versus arrival time and multi-event keys

The CDC files are deliberately shuffled. Sort by event time rather than file order or arrival order when constructing a historical timeline.

In [ ]:
# Inspect late-arriving events without implying that late arrival is always invalid.
for domain, df in valid_events.items():
    delayed = (df.withColumn("delay_hours",
        (F.unix_timestamp("arrived_at") - F.unix_timestamp("event_at")) / F.lit(3600.0)))
    print(domain, "arrival delays > 24h:", delayed.filter("delay_hours > 24").count())
    display(delayed.select("event_id", "operation", "event_at", "arrived_at", "delay_hours")
            .orderBy(F.desc("delay_hours")).limit(6))

# Multiple UNIQUE changes for one customer are not replay duplicates.
display(valid_events["customers"].groupBy("customer_id")
        .count().filter("count > 1").orderBy(F.desc("count"), "customer_id"))

## STEP 09 — Initialize three LAB 06-owned Delta targets, without modifying previous labs

- Copy the historical customer dimension **once**, including pre-existing LAB 03 versions.
- Copy the trusted current order and payment tables **once**.
- Add CDC-specific metadata fields. Existing LAB 03 and LAB 05 datasets remain unchanged.
- If LAB 06 target tables already exist, **reuse** them: never overwrite their history.

This is a controlled one-time bootstrap, not a continuously synchronized clone.

In [ ]:
# LAB 06 owns these new objects. Their initialization is safe to rerun.
BASE_CDC_COLUMNS = ["cdc_event_id", "cdc_event_timestamp", "cdc_source_sequence",
                    "cdc_ingestion_timestamp"]

if not spark.catalog.tableExists(CUSTOMERS_TARGET):
    seed = (spark.table(BASE_CUSTOMERS)
        .withColumn("is_deleted", F.lit(False))
        .withColumn("cdc_event_id", F.lit(None).cast("string"))
        .withColumn("cdc_event_timestamp", F.lit(None).cast("timestamp"))
        .withColumn("cdc_source_sequence", F.lit(None).cast("int"))
        .withColumn("cdc_ingestion_timestamp", F.lit(None).cast("timestamp")))
    seed.write.format("delta").mode("errorifexists").saveAsTable(CUSTOMERS_TARGET)
    print("CREATED", CUSTOMERS_TARGET)
else:
    print("REUSED", CUSTOMERS_TARGET)

for original, target in [(BASE_ORDERS, ORDERS_TARGET), (BASE_PAYMENTS, PAYMENTS_TARGET)]:
    if not spark.catalog.tableExists(target):
        seed = (spark.table(original)
            .withColumn("is_deleted", F.lit(False))
            .withColumn("cdc_event_id", F.lit(None).cast("string"))
            .withColumn("cdc_event_timestamp", F.lit(None).cast("timestamp"))
            .withColumn("cdc_source_sequence", F.lit(None).cast("int"))
            .withColumn("cdc_ingestion_timestamp", F.lit(None).cast("timestamp")))
        seed.write.format("delta").mode("errorifexists").saveAsTable(target)
        print("CREATED", target)
    else:
        print("REUSED", target)

for target in (CUSTOMERS_TARGET, ORDERS_TARGET, PAYMENTS_TARGET):
    assert set(BASE_CDC_COLUMNS + ["is_deleted"]) <= set(spark.table(target).columns)
    assert spark.sql(f"DESCRIBE DETAIL {target}").select("format").first()[0] == "delta"

## STEP 10 — Validate CDC keys against the bootstrapped current state

An UPDATE or DELETE for an unknown customer cannot be applied as a full customer row. An INSERT for a customer that already exists would require a separate reactivation policy.

In [ ]:
# Classify key semantics BEFORE applying the SCD2 timeline.
customer_current = (spark.table(CUSTOMERS_TARGET)
    .filter("is_current = true")
    .select("customer_id", F.col("effective_start").alias("__current_start"))
    .withColumn("__key_exists", F.lit(True)))
# A previously applied INSERT is a replay, not a new INSERT conflict.
customer_keyed = valid_events["customers"].join(customer_current, "customer_id", "left")
customer_keyed = (customer_keyed
    .withColumn("key_problem", F.when(
        (F.col("operation") == "INSERT") & F.col("__key_exists").isNotNull() &
        (F.col("event_at") > F.col("__current_start")),
        F.lit("INSERT_ALREADY_EXISTS")
    ).when(
        F.col("operation").isin("UPDATE", "DELETE") & F.col("__key_exists").isNull(),
        F.lit("CHANGE_WITHOUT_BASE_ROW")
    )))
customer_key_rejected = customer_keyed.filter("key_problem IS NOT NULL")
customer_key_valid = customer_keyed.filter("key_problem IS NULL").drop("__key_exists", "__current_start", "key_problem")
print("Customer events with incompatible business keys:", customer_key_rejected.count())
display(customer_key_rejected.select("event_id", "operation", "customer_id", "key_problem").limit(12))

## STEP 11 — Customer SCD2: prepare **sparse PATCH** events and baseline rows

We will reconstruct the complete state after each incoming event using Spark window functions. The baseline is the existing **current** customer row, and null CDC patch attributes inherit the last known value.

**Limitation:** Empty CDC values mean *not supplied*, so this source contract cannot express a deliberate `SET email = NULL`. An enterprise-grade CDC feed should include field-presence semantics or a complete after-image.

In [ ]:
# Keep the exact original LAB 03 tracked-attribute contract and hashing order.
TRACKED = ["first_name", "last_name", "email", "phone", "date_of_birth",
           "country", "city", "postal_code", "customer_segment", "marketing_consent"]
CUSTOMER_BUSINESS = ["customer_id", "first_name", "last_name", "email", "phone",
                     "date_of_birth", "country", "city", "postal_code", "customer_segment",
                     "marketing_consent", "created_at", "updated_at", "source_system", "notes",
                     "ingestion_timestamp", "ingestion_batch_id", "source_file", "source_repository",
                     "silver_processed_timestamp"]
HISTORY_BASE_COLS = CUSTOMER_BUSINESS + ["record_hash", "effective_start", "effective_end",
                                         "is_current", "version", "is_deleted"] + BASE_CDC_COLUMNS

assert set(HISTORY_BASE_COLS) == set(spark.table(CUSTOMERS_TARGET).columns), (
    "LAB 03 historical schema differs from the expected LAB 06 contract"
)

# Parse patch fields only when supplied. Normalize country/email and preserve typed values.
e = customer_key_valid
patch = (e
    .withColumn("first_name", F.trim("first_name"))
    .withColumn("last_name", F.trim("last_name"))
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("phone", F.trim("phone"))
    .withColumn("city", F.trim("city"))
    .withColumn("country", F.upper(F.trim("country")))
    .withColumn("customer_segment", F.initcap(F.lower(F.trim("customer_segment"))))
    .withColumn("date_of_birth", F.expr("try_cast(date_of_birth AS DATE)"))
    .withColumn("marketing_consent", F.expr("try_cast(marketing_consent AS BOOLEAN)"))
    .withColumn("created_at", F.expr("try_cast(created_at AS TIMESTAMP)"))
    .withColumn("updated_at", F.col("event_at")))

# Avoid assigning NULL to fields that a partial CDC message does not include.
# The 'notes' field documents this *CDC event*, not a customer address field.
PATCH_FIELDS = TRACKED + ["created_at", "updated_at", "source_system", "notes",
                          "ingestion_timestamp", "ingestion_batch_id", "source_file",
                          "source_repository", "silver_processed_timestamp"]

# Build an event-shaped row with correctly typed values and CDC control columns.
patch = (patch
    .withColumn("postal_code", F.lit(None).cast("string"))
    .withColumn("ingestion_timestamp", F.col("arrived_at"))
    .withColumn("ingestion_batch_id", F.lit(BATCH_ID))
    .withColumn("source_file", F.lit(CDC["customers"]["filename"]))
    .withColumn("source_repository", F.lit(REPO))
    .withColumn("silver_processed_timestamp", F.current_timestamp())
    .withColumn("cdc_event_id", F.col("event_id"))
    .withColumn("cdc_event_timestamp", F.col("event_at"))
    .withColumn("cdc_source_sequence", F.col("event_sequence"))
    .withColumn("cdc_ingestion_timestamp", F.col("arrived_at"))
    .withColumn("event_kind", F.lit("EVENT")))

## STEP 12 — Customer SCD2: reject genuinely stale events (relative to previously committed batches)

Within a **single new file**, a late-arriving event can be reconstructed correctly by sorting *all* events by event time. But after a later version has already been committed from another batch, blindly applying an older event would corrupt the timeline. We explicitly classify that as `STALE_VS_COMMITTED_VERSION` rather than quietly rewriting the past.

For complete multi-batch out-of-order historical reconstruction, evaluate **Lakeflow AUTO CDC** with `SEQUENCE BY` and `STORED AS SCD TYPE 2`.

In [ ]:
# Select latest already-committed active version per business key.
current_versions = spark.table(CUSTOMERS_TARGET).filter("is_current = true")
watermarks = current_versions.select(
    "customer_id",
    F.col("effective_start").alias("__current_effective_start")
)
patch_with_mark = patch.join(watermarks, "customer_id", "left")
customer_stale = patch_with_mark.filter(
    F.col("__current_effective_start").isNotNull()
    & (F.col("event_at") <= F.col("__current_effective_start"))
)
customer_fresh = patch_with_mark.filter(
    F.col("__current_effective_start").isNull()
    | (F.col("event_at") > F.col("__current_effective_start"))
).drop("__current_effective_start")

# Same timestamp for different event IDs would need a more precise effective-ordering
# model than a TIMESTAMP-only half-open validity range. Fail fast if encountered.
colliding = (customer_fresh.groupBy("customer_id", "event_at")
    .count().filter("count > 1"))
assert colliding.count() == 0, "Equal-time customer events require an explicit source ordering policy"

# Snapshot counts BEFORE the target is changed; Spark DataFrames are lazily evaluated.
CUSTOMER_INPUT_METRICS = {
    "key_rejected": customer_key_rejected.count(),
    "stale_vs_committed": customer_stale.count(),
    "fresh": customer_fresh.count(),
}
print("Customer event classification BEFORE MERGE:", CUSTOMER_INPUT_METRICS)

## STEP 13 — Customer SCD2: reconstruct each full row using window `last(..., ignorenulls=True)`

This preserves intermediate event-time versions (including four delayed updates) and supports multiple UPDATEs followed by DELETE for one key. The baseline is not inserted again.

In [ ]:
# Existing current rows supply the state before the first newly accepted CDC event.
affected = customer_fresh.select("customer_id").distinct()
base = current_versions.join(affected, "customer_id", "inner")

# A separate synthetic baseline row is needed even for a brand-new INSERT key.
missing_baseline = (affected.join(base.select("customer_id"), "customer_id", "left_anti")
    .select("customer_id"))
empty_seed = (missing_baseline.withColumn("effective_start", F.to_timestamp(F.lit("1900-01-01")))
    .withColumn("version", F.lit(0))
    .withColumn("is_deleted", F.lit(False)))

# Union baseline + event rows with matching types; all other attributes are NULL.
# The extra CDC timestamp/sequence columns are part of the state contract.
STATE_COLS = [c for c in HISTORY_BASE_COLS if c not in (
    "record_hash", "effective_end", "is_current", "version", "is_deleted", "effective_start")]

baseline_real = (base
    .withColumn("event_at", F.col("effective_start"))
    .withColumn("event_sequence", F.lit(-1))
    .withColumn("event_id", F.lit("BASELINE"))
    .withColumn("operation", F.lit("BASELINE"))
    .withColumn("event_kind", F.lit("BASELINE"))
    .select(*STATE_COLS, "event_at", "event_sequence", "event_id", "operation", "event_kind",
            "version", "is_deleted"))

# Use explicit types borrowed from the real dimension schema when fabricating an empty seed.
target_types = {field.name: field.dataType for field in spark.table(CUSTOMERS_TARGET).schema.fields}
for colname in STATE_COLS:
    if colname not in empty_seed.columns:
        empty_seed = empty_seed.withColumn(colname, F.lit(None).cast(target_types[colname]))
empty_seed = (empty_seed
    .withColumn("event_at", F.to_timestamp(F.lit("1900-01-01")))
    .withColumn("event_sequence", F.lit(-1))
    .withColumn("event_id", F.lit("BASELINE"))
    .withColumn("operation", F.lit("BASELINE"))
    .withColumn("event_kind", F.lit("BASELINE"))
    .select(*STATE_COLS, "event_at", "event_sequence", "event_id", "operation", "event_kind",
            "version", "is_deleted"))

baseline = baseline_real.unionByName(empty_seed)

# Event rows provide only fields included in the source patch; other fields are inherited.
updates = (customer_fresh
    .withColumn("version", F.lit(None).cast("int"))
    .withColumn("is_deleted", F.lit(None).cast("boolean"))
    .select(*STATE_COLS, "event_at", "event_sequence", "event_id", "operation", "event_kind",
            "version", "is_deleted"))

combined = baseline.unionByName(updates)
order_window = (Window.partitionBy("customer_id")
    .orderBy(F.col("event_at").asc(), F.col("event_sequence").asc(), F.col("event_id").asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow))

# Rebuild each field as the last non-NULL value on that customer's event timeline.
# Do not carry forward event-ID metadata; it belongs to the current EVENT only.
INHERITED_FIELDS = [x for x in STATE_COLS if x not in (
    "customer_id", "cdc_event_id", "cdc_event_timestamp", "cdc_source_sequence", "cdc_ingestion_timestamp")]
full_timeline = combined
for colname in INHERITED_FIELDS:
    full_timeline = full_timeline.withColumn(
        colname, F.last(F.col(colname), ignorenulls=True).over(order_window))

# Preserve the base version number for sequential SCD2 version assignment.
full_timeline = full_timeline.withColumn(
    "__base_version", F.first("version", ignorenulls=True).over(
        Window.partitionBy("customer_id").orderBy("event_at", "event_sequence", "event_id")
        .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))

print("Customer source timeline reconstructed; no historical Delta changes yet.")

## STEP 14 — Customer SCD2: detect meaningful state transitions and date every version

Use the same tracked attributes as LAB 03. An UPDATE with no change to tracked attributes need not create an SCD2 version. A DELETE must create a tombstone version and end the preceding state.

In [ ]:
# Hash the fully inherited customer state, not the sparse source PATCH itself.
full_timeline = full_timeline.withColumn(
    "record_hash", F.sha2(F.to_json(F.struct(*[F.col(c) for c in TRACKED]),
                            options={"ignoreNullFields": "false"}), 256)
)
prev_window = Window.partitionBy("customer_id").orderBy("event_at", "event_sequence", "event_id")

# For a DELETE row, preserve the last known business attributes but mark the row deleted.
full_timeline = (full_timeline
    .withColumn("__prev_hash", F.lag("record_hash").over(prev_window))
    .withColumn("__prev_deleted", F.lag(F.coalesce("is_deleted", F.lit(False))).over(prev_window)))

# Keep meaningful versions, including a tombstone after any non-deleted version.
transition = (full_timeline.filter("event_kind = 'EVENT'")
    .filter(F.col("__prev_hash").isNull()
            | (F.col("record_hash") != F.col("__prev_hash"))
            | (F.col("operation") == "DELETE")))

# Recompute contiguous version numbers and end timestamps AFTER no-op removal.
version_window = Window.partitionBy("customer_id").orderBy("event_at", "event_sequence", "event_id")
transition = (transition
    .withColumn("__rn", F.row_number().over(version_window))
    .withColumn("version", (F.col("__base_version") + F.col("__rn")).cast("int"))
    .withColumn("effective_start", F.col("event_at"))
    .withColumn("effective_end", F.lead("event_at").over(version_window))
    .withColumn("is_current", F.col("effective_end").isNull())
    .withColumn("is_deleted", F.col("operation") == "DELETE")
    .withColumn("cdc_event_id", F.col("event_id"))
    .withColumn("cdc_event_timestamp", F.col("event_at"))
    .withColumn("cdc_source_sequence", F.col("event_sequence"))
    .withColumn("cdc_ingestion_timestamp", F.col("ingestion_timestamp"))
)

# A later non-delete version of the same key must not follow its tombstone without
# an explicit reactivation policy. The fixture contains only terminal deletes.
after_delete = (transition.withColumn("__previous_delete", F.lag("is_deleted").over(version_window))
    .filter(F.col("__previous_delete") == True))
assert after_delete.count() == 0, "CDC reactivation after DELETE requires a business policy"

print("Meaningful customer SCD2 versions:", transition.count())
display(transition.select("customer_id", "event_id", "operation", "city", "version",
                          "effective_start", "effective_end", "is_current", "is_deleted")
        .orderBy("customer_id", "version").limit(25))

## STEP 15 — Customer SCD2: implement a **staged** transactional Delta MERGE

For each affected existing customer, one matched row expires the old current version at the **first** new event time. Each meaningful event becomes one inserted version. The final inserted version stays current, even if it is a soft-deleted tombstone.

**One transaction per target Delta table**, with staging materialized independently of the target to avoid reading a changing target during MERGE. No cross-table atomic transaction is claimed.

In [ ]:
# A staged MERGE can insert multiple historical versions per key in one commit.
# Only one row per key is allowed to MATCH the current target row.
STAGE_CUST = f"{SILVER}._lab06_customers_merge_stage"
HISTORY_COLS = spark.table(CUSTOMERS_TARGET).columns

changes_to_insert = transition.select(*HISTORY_COLS)
first_events = (changes_to_insert.withColumn(
    "__first", F.row_number().over(Window.partitionBy("customer_id")
        .orderBy("effective_start", "version"))).filter("__first = 1"))
existing_keys = current_versions.select("customer_id")

# Explicit expiration rows are created only for keys with an existing active version.
expiration = (first_events.join(existing_keys, "customer_id", "inner")
    .select(F.col("customer_id").alias("merge_key"),
            *[F.col(c) for c in HISTORY_COLS]))
insertions = changes_to_insert.select(
    F.lit(None).cast("string").alias("merge_key"),
    *[F.col(c) for c in HISTORY_COLS])
customer_stage = expiration.unionByName(insertions)

customer_staged_count = customer_stage.count()
if customer_staged_count:
    # Staging is an internal LAB 06 work table and is safe to replace.
    (customer_stage.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(STAGE_CUST))
    stable_stage = spark.table(STAGE_CUST)
    DeltaTable.forName(spark, CUSTOMERS_TARGET).alias("trg").merge(
        stable_stage.alias("src"),
        "trg.customer_id = src.merge_key AND trg.is_current = true"
    ).whenMatchedUpdate(
        condition="src.effective_start > trg.effective_start",
        set={"effective_end": "src.effective_start", "is_current": "false"}
    ).whenNotMatchedInsert(
        condition="src.merge_key IS NULL",
        values={c: f"src.{c}" for c in HISTORY_COLS}
    ).execute()
    print("Customer historical MERGE committed:", customer_staged_count, "stage rows")
else:
    print("No new customer transitions: MERGE skipped (replay-safe).")

## STEP 16 — Validate SCD2 history and publish active customer views

A tombstone row is `is_current=true AND is_deleted=true`; an active customer is current **and not deleted**. `effective_end` is exclusive.

In [ ]:
# Publish the current, non-deleted customer projection without replacing LAB 03's view.
spark.sql(f"""
CREATE OR REPLACE VIEW {CUSTOMERS_VIEW} AS
SELECT * FROM {CUSTOMERS_TARGET}
WHERE is_current = true AND is_deleted = false
""")

h = spark.table(CUSTOMERS_TARGET)
assert h.filter("is_current = true").groupBy("customer_id").count().filter("count > 1").count() == 0
assert h.filter("is_current = true AND effective_end IS NOT NULL").count() == 0
assert h.filter("is_current = false AND effective_end IS NULL").count() == 0
assert h.filter("effective_end IS NOT NULL AND effective_end <= effective_start").count() == 0
assert h.filter("customer_id IS NULL").count() == 0

# Find historical overlap using consecutive effective intervals.
w = Window.partitionBy("customer_id").orderBy("effective_start", "version")
overlap = (h.withColumn("prev_end", F.lag("effective_end").over(w))
    .withColumn("prev_start", F.lag("effective_start").over(w))
    .filter(F.col("prev_start").isNotNull() &
        (F.col("prev_end").isNull() | (F.col("effective_start") < F.col("prev_end")))))
assert overlap.count() == 0, "Overlapping historical intervals detected"

print("SCD2 versions:", h.count())
print("Non-deleted current customers:", spark.table(CUSTOMERS_VIEW).count())
print("Current deletion tombstones:", h.filter("is_current AND is_deleted").count())
display(h.filter("cdc_event_id IS NOT NULL")
        .select("customer_id", "city", "effective_start", "effective_end", "is_current",
                "is_deleted", "version", "cdc_event_id")
        .orderBy("customer_id", "version").limit(30))

## STEP 17 — Orders CDC: choose latest event **per order** for a current-state upsert

Orders use SCD1-like current-state processing here; we do not pretend to preserve every intermediate order status. Older in-file changes are accounted for as **superseded within the batch**, not as deleted source records.

A `DELETE` becomes `is_deleted = true` rather than removing the order or its line items. This gives downstream analytics an explicit option to exclude soft-deleted orders.

In [ ]:
# Keep the latest unique event per business key for a current-state materialization.
def latest_per_key(df, key):
    ordering = Window.partitionBy(key).orderBy(
        F.col("event_at").desc(), F.col("event_sequence").desc(), F.col("event_id").desc())
    ranked = df.withColumn("__rank", F.row_number().over(ordering))
    return ranked.filter("__rank = 1").drop("__rank"), ranked.filter("__rank > 1").drop("__rank")

orders_latest, orders_superseded = latest_per_key(valid_events["orders"], "order_id")
print("Orders canonical:", valid_events["orders"].count(),
      "latest per order:", orders_latest.count(),
      "older unique in-file events:", orders_superseded.count())

order_target = spark.table(ORDERS_TARGET)
order_refs = order_target.select(
    "order_id", F.col("updated_at").alias("__base_updated"),
    F.col("cdc_event_timestamp").alias("__prior_cdc_event"))
orders_compared = orders_latest.join(order_refs, "order_id", "left")
orders_unknown = orders_compared.filter("__base_updated IS NULL AND __prior_cdc_event IS NULL")
orders_known = orders_compared.filter("__base_updated IS NOT NULL OR __prior_cdc_event IS NOT NULL")

# If a CDC event predates the current source snapshot, do not overwrite newer state.
order_watermark = F.greatest(F.col("__base_updated"), F.col("__prior_cdc_event"))
orders_stale = orders_known.filter(F.col("event_at") <= order_watermark)
orders_actionable = orders_known.filter(F.col("event_at") > order_watermark)
ORDER_INPUT_METRICS = {
    "superseded_in_file": orders_superseded.count(),
    "unknown_key": orders_unknown.count(),
    "stale_vs_committed": orders_stale.count(),
    "actionable": orders_actionable.count(),
}
print("Order event classification BEFORE MERGE:", ORDER_INPUT_METRICS)

## STEP 18 — Execute conditional `MERGE` for orders

**Important:** `new_total_amount_eur` is blank in this fixture. Keep the existing amount; do not interpret blank as zero. Use a materialized stage and a stricter target-side timestamp predicate.

In [ ]:
ORDER_STAGE = f"{SILVER}._lab06_orders_merge_stage"

order_stage = orders_actionable.select(
    "event_id", "order_id", "operation", "event_at", "event_sequence", "arrived_at",
    "new_order_status", "new_amount_parsed"
)
if order_stage.count():
    (order_stage.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(ORDER_STAGE))
    src = spark.table(ORDER_STAGE)
    DeltaTable.forName(spark, ORDERS_TARGET).alias("trg").merge(
        src.alias("src"), "trg.order_id = src.order_id"
    ).whenMatchedUpdate(
        condition=("src.event_at > COALESCE(trg.cdc_event_timestamp, trg.updated_at) "
                   "AND src.event_at > trg.updated_at"),
        set={
            "order_status": "CASE WHEN src.operation = 'UPDATE' THEN src.new_order_status ELSE trg.order_status END",
            "total_amount_eur": "COALESCE(src.new_amount_parsed, trg.total_amount_eur)",
            "is_deleted": "src.operation = 'DELETE'",
            "updated_at": "src.event_at",
            "cdc_event_id": "src.event_id",
            "cdc_event_timestamp": "src.event_at",
            "cdc_source_sequence": "src.event_sequence",
            "cdc_ingestion_timestamp": "src.arrived_at",
        }
    ).execute()
    print("Order MERGE committed")
else:
    print("No new order events: MERGE skipped.")

spark.sql(f"CREATE OR REPLACE VIEW {ORDERS_VIEW} AS "
          f"SELECT * FROM {ORDERS_TARGET} WHERE is_deleted = false")

## STEP 19 — Payments CDC: current-state changes are NOT new money movements

The payment gateway feed contains status updates to **existing** `payment_id`s. A change from PENDING to CAPTURED is an UPDATE to the same payment, not a new charge. Refunds already exist as separate signed payment records in LAB 05; **do not create another refund from a status change**.

In [ ]:
# Payment keys reference existing payment attempts, not orders as the primary key.
payments_latest, payments_superseded = latest_per_key(valid_events["payments"], "payment_id")
payment_target = spark.table(PAYMENTS_TARGET)
payment_refs = payment_target.select(
    "payment_id", F.col("order_id").alias("__target_order"),
    F.col("updated_at").alias("__base_updated"),
    F.col("cdc_event_timestamp").alias("__prior_cdc_event"))
payments_compared = payments_latest.join(payment_refs, "payment_id", "left")
payments_unknown = payments_compared.filter("__target_order IS NULL")
payments_wrong_order = payments_compared.filter(
    F.col("__target_order").isNotNull() &
    (F.col("__target_order") != F.col("order_id")))
payments_known = payments_compared.filter(
    F.col("__target_order").isNotNull() &
    (F.col("__target_order") == F.col("order_id")))
payment_watermark = F.greatest(F.col("__base_updated"), F.col("__prior_cdc_event"))
payments_stale = payments_known.filter(F.col("event_at") <= payment_watermark)
payments_actionable = payments_known.filter(F.col("event_at") > payment_watermark)
PAYMENT_INPUT_METRICS = {
    "superseded_in_file": payments_superseded.count(),
    "unknown_key": payments_unknown.count(),
    "mismatched_order_id": payments_wrong_order.count(),
    "stale_vs_committed": payments_stale.count(),
    "actionable": payments_actionable.count(),
}
print("Payment event classification BEFORE MERGE:", PAYMENT_INPUT_METRICS)

## STEP 20 — Execute conditional `MERGE` for payments

Keep the original monetary `DECIMAL` and gateway transaction identity. Update `payment_status` and optionally the amount when explicitly supplied.

In [ ]:
PAYMENT_STAGE = f"{SILVER}._lab06_payments_merge_stage"

payment_stage = payments_actionable.select(
    "event_id", "payment_id", "order_id", "event_at", "event_sequence",
    "arrived_at", "new_payment_status", "new_amount_parsed"
)
if payment_stage.count():
    (payment_stage.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(PAYMENT_STAGE))
    src = spark.table(PAYMENT_STAGE)
    DeltaTable.forName(spark, PAYMENTS_TARGET).alias("trg").merge(
        src.alias("src"),
        "trg.payment_id = src.payment_id AND trg.order_id = src.order_id"
    ).whenMatchedUpdate(
        condition=("src.event_at > COALESCE(trg.cdc_event_timestamp, trg.updated_at) "
                   "AND src.event_at > trg.updated_at"),
        set={
            "payment_status": "src.new_payment_status",
            "amount_eur": "COALESCE(src.new_amount_parsed, trg.amount_eur)",
            "updated_at": "src.event_at",
            "cdc_event_id": "src.event_id",
            "cdc_event_timestamp": "src.event_at",
            "cdc_source_sequence": "src.event_sequence",
            "cdc_ingestion_timestamp": "src.arrived_at",
        }
    ).execute()
    print("Payment MERGE committed")
else:
    print("No new payment events: MERGE skipped.")

spark.sql(f"CREATE OR REPLACE VIEW {PAYMENTS_VIEW} AS "
          f"SELECT * FROM {PAYMENTS_TARGET} WHERE is_deleted = false")

## STEP 21 — CDC accounting: distinguish replays, rejects, stale changes, and superseded events

Unlike a source-row to current-table equality, the following is a **per-domain event accounting**. An event can be successfully processed without adding a new row (orders/payments UPDATE) or can produce several historical rows (customer SCD2).

In [ ]:
# These statuses are mutually exclusive within each domain after event-id deduplication.
cust_event_accounts = {
    "source": spark.table(CDC["customers"]["bronze"]).count(),
    "replays": replayed["customers"].count(),
    "quality_rejected": invalid_events["customers"].count(),
    **CUSTOMER_INPUT_METRICS,
}
order_event_accounts = {
    "source": spark.table(CDC["orders"]["bronze"]).count(),
    "replays": replayed["orders"].count(),
    "quality_rejected": invalid_events["orders"].count(),
    **ORDER_INPUT_METRICS,
}
payment_event_accounts = {
    "source": spark.table(CDC["payments"]["bronze"]).count(),
    "replays": replayed["payments"].count(),
    "quality_rejected": invalid_events["payments"].count(),
    **PAYMENT_INPUT_METRICS,
}
assert cust_event_accounts["source"] == sum(
    v for k,v in cust_event_accounts.items() if k != "source")
assert order_event_accounts["source"] == sum(
    v for k,v in order_event_accounts.items() if k != "source")
assert payment_event_accounts["source"] == sum(
    v for k,v in payment_event_accounts.items() if k != "source")

for domain, record in [("customers", cust_event_accounts), ("orders", order_event_accounts),
                       ("payments", payment_event_accounts)]:
    print(domain, record)

# Persist an append-only per-run audit summary? For this single-file laboratory we create a
# stable first-run report table once and never overwrite it during idempotency replay.
CDC_AUDIT_TABLE = f"{SILVER}.cdc_batch_audit_lab06"
metrics_rows = [
    (domain, metric, int(count), BATCH_ID)
    for domain, report in [
        ("customers", cust_event_accounts),
        ("orders", order_event_accounts),
        ("payments", payment_event_accounts),
    ]
    for metric, count in report.items()
]
metrics_df = spark.createDataFrame(metrics_rows,
    schema="domain STRING, metric STRING, event_count LONG, batch_id STRING")
if not spark.catalog.tableExists(CDC_AUDIT_TABLE):
    metrics_df.write.format("delta").mode("errorifexists").saveAsTable(CDC_AUDIT_TABLE)
    print("Created first-run CDC batch audit:", CDC_AUDIT_TABLE)
else:
    print("Preserved existing CDC audit (original run):", CDC_AUDIT_TABLE)
display(spark.table(CDC_AUDIT_TABLE).orderBy("domain", "metric"))

## STEP 22 — Demonstrate idempotency by **recomputing** against committed targets

Do not replay a stale staging DataFrame computed before the first MERGE. Re-run STEP 10 through STEP 20 in order: newly committed target watermarks must be read again. Then verify that the target row counts and content fingerprints have not changed.

In [ ]:
# Persist simple post-commit checkpoints as Python values for the replay exercise.
CHECKPOINTS = {
    "customers_versions": spark.table(CUSTOMERS_TARGET).count(),
    "orders_rows": spark.table(ORDERS_TARGET).count(),
    "payments_rows": spark.table(PAYMENTS_TARGET).count(),
    "customers_cdc_ids": spark.table(CUSTOMERS_TARGET).filter("cdc_event_id IS NOT NULL")
                        .select("cdc_event_id").distinct().count(),
    "orders_cdc_ids": spark.table(ORDERS_TARGET).filter("cdc_event_id IS NOT NULL")
                     .select("cdc_event_id").distinct().count(),
    "payments_cdc_ids": spark.table(PAYMENTS_TARGET).filter("cdc_event_id IS NOT NULL")
                       .select("cdc_event_id").distinct().count(),
}
print("Snapshot checkpoints captured:", CHECKPOINTS)
print("Replay procedure: execute STEP 10–20 once more, then run STEP 23.")

## STEP 23 — Verify replay safety and historical integrity

After re-running STEP 10–20, the expected results are:

- No additional customer-history versions.
- No newly inserted orders or payments.
- No duplicate current customer versions.
- No overlapping customer-effective intervals.
- Stable event IDs in the target tables.

Note: This does not by itself prove cross-table exactly-once semantics, streaming checkpoint recovery, or concurrent-writer safety.

In [ ]:
# Compare against the prior explicit checkpoint (works after a manual replay).
CURRENT_CHECKPOINTS = {
    "customers_versions": spark.table(CUSTOMERS_TARGET).count(),
    "orders_rows": spark.table(ORDERS_TARGET).count(),
    "payments_rows": spark.table(PAYMENTS_TARGET).count(),
    "customers_cdc_ids": spark.table(CUSTOMERS_TARGET).filter("cdc_event_id IS NOT NULL")
                        .select("cdc_event_id").distinct().count(),
    "orders_cdc_ids": spark.table(ORDERS_TARGET).filter("cdc_event_id IS NOT NULL")
                     .select("cdc_event_id").distinct().count(),
    "payments_cdc_ids": spark.table(PAYMENTS_TARGET).filter("cdc_event_id IS NOT NULL")
                       .select("cdc_event_id").distinct().count(),
}
assert CURRENT_CHECKPOINTS == CHECKPOINTS, (
    f"Replay changed CDC targets: before={CHECKPOINTS}, after={CURRENT_CHECKPOINTS}"
)
assert (spark.table(CUSTOMERS_TARGET).filter("is_current = true")
        .groupBy("customer_id").count().filter("count > 1").count()) == 0
print("PASS: CDC targets unchanged by replay; customer current-state uniqueness holds.")

## STEP 24 — Analyze soft deletes, payment status changes, and SCD2 historical lookups

Use these queries to tell a coherent business story during a Solutions Architect demonstration. Deleted customers are excluded from the active-customer view but remain present in historical analysis.

In [ ]:
# Business-facing diagnostics: never treat CDC replay as additional money.
print("Customer tombstones:")
display(spark.table(CUSTOMERS_TARGET).filter("is_current AND is_deleted")
        .select("customer_id", "effective_start", "cdc_event_id").limit(20))

print("Order statuses in CDC-updated current snapshot:")
display(spark.table(ORDERS_TARGET).groupBy("order_status", "is_deleted").count()
        .orderBy("order_status", "is_deleted"))

print("Payment statuses and charge/refund types after CDC:")
display(spark.table(PAYMENTS_TARGET).groupBy("payment_type", "payment_status")
        .agg(F.count("*").alias("rows"), F.sum("amount_eur").alias("total_amount_eur"))
        .orderBy("payment_type", "payment_status"))

In [ ]:
# Historical, half-open interval lookup: event_date >= start and < end.
# Prefer a real updated customer if at least one event was committed.
example = (spark.table(CUSTOMERS_TARGET).filter("cdc_event_id IS NOT NULL")
    .orderBy("customer_id", "version").select("customer_id").first())
if example:
    cid = example["customer_id"]
    print("Historical versions of:", cid)
    display(spark.table(CUSTOMERS_TARGET).filter(F.col("customer_id") == cid)
            .select("customer_id", "city", "customer_segment", "effective_start",
                    "effective_end", "is_current", "is_deleted", "version")
            .orderBy("effective_start"))
    # As-of query returns exactly one version for a chosen event time.
    middle = (spark.table(CUSTOMERS_TARGET)
        .filter(F.col("customer_id") == cid)
        .orderBy("effective_start").select("effective_start").first()[0])
    as_of = spark.table(CUSTOMERS_TARGET).filter(
        (F.col("customer_id") == cid) &
        (F.col("effective_start") <= F.lit(middle)) &
        (F.col("effective_end").isNull() | (F.col("effective_end") > F.lit(middle))))
    assert as_of.count() == 1, "An as-of timestamp should match exactly one version"
    display(as_of.select("customer_id", "city", "effective_start", "effective_end", "is_deleted"))
else:
    print("No new customer events were committed in this run.")

## STEP 25 — Verify Unity Catalog objects and Delta transaction history

Use `DESCRIBE HISTORY` to show actual Delta commits. The audit demonstrates atomicity **per table**, not across the three separate MERGEs.

In [ ]:
# Reuse independent LAB 06 targets and inspect metadata.
display(spark.sql(f"SHOW TABLES IN {SILVER}"))
display(spark.sql(f"SHOW TABLES IN {BRONZE}"))
for tbl in (CUSTOMERS_TARGET, ORDERS_TARGET, PAYMENTS_TARGET):
    print("DELTA TABLE:", tbl)
    display(spark.sql(f"DESCRIBE DETAIL {tbl}").select("format", "location", "numFiles"))
    display(spark.sql(f"DESCRIBE HISTORY {tbl}").select("version", "timestamp", "operation").limit(8))

## STEP 26 — Final LAB 06 acceptance criteria

Success means:

1. All three exact GitHub CDC files reached Bronze (680 rows).
2. Event-ID replay detection and event quality validation passed.
3. Customer changes were applied to LAB 06's own SCD2 target (with history and tombstones).
4. Orders and payment statuses were conditionally MERGEd into LAB 06 current-state tables.
5. No LAB 03/05 source tables were changed by this notebook.
6. Every CDC input row is explained in the per-domain event accounting.

In [ ]:
assert sum(spark.table(x["bronze"]).count() for x in CDC.values()) == 680
assert spark.table(CUSTOMERS_TARGET).filter("customer_id IS NULL").count() == 0
assert spark.table(CUSTOMERS_TARGET).filter("is_current = true").groupBy("customer_id") \
    .count().filter("count > 1").count() == 0
assert spark.table(ORDERS_TARGET).groupBy("order_id").count().filter("count > 1").count() == 0
assert spark.table(PAYMENTS_TARGET).groupBy("payment_id").count().filter("count > 1").count() == 0

print("=" * 72)
print("LAB 06 — DELTA MERGE & CDC: VALIDATION PASSED")
print("=" * 72)
print("Raw source events: 680")
print("Canonical events:", sum(canonical[d].count() for d in CDC))
print("Source replay rows:", sum(replayed[d].count() for d in CDC))
print("Current active customers:", spark.table(CUSTOMERS_VIEW).count())
print("Customer history rows:", spark.table(CUSTOMERS_TARGET).count())
print("Order snapshot rows:", spark.table(ORDERS_TARGET).count())
print("Payment snapshot rows:", spark.table(PAYMENTS_TARGET).count())
print("LAB 03 and LAB 05 tables preserved by design.")
print("=" * 72)

## STEP 27 — Architecture limitations and the production alternative

**This exercise is deliberately not a production CDC engine.**

- **Partial updates:** empty CRM columns are treated as "not sent." A production feed should distinguish "missing" from "explicitly set NULL".
- **Deletes:** tombstones are logical. A privacy deletion request may require a separate compliant physical-erasure workflow.
- **Event-time ordering:** this lab reconstructs multiple events **within the same initial file**. A late earlier event received *after a more recent batch was already committed* is rejected as stale rather than inserted into a past validity interval.
- **Order/payment history:** we maintain only current-state values, not every intermediate status. This is SCD1-like semantics, intentionally different from customers SCD2.
- **Concurrency:** the lab assumes one writer. It does not implement transactional coordination across customer/order/payment targets or a persistent exactly-once streaming offset store.
- **Business-time accuracy:** a source system must provide a reliable ordering/version field; this fixture's `source_sequence` is an illustrative extra signal, not a universal log sequence.
- **Gold correctness:** downstream revenue must explicitly decide whether to exclude cancelled, returned, or deleted orders and how refunds affect net revenue.
- **Historical customers:** LAB 03's bootstrap may already have been reduced to current snapshots. LAB 06 preserves history from its copied baseline onward; it cannot reconstruct information that was not available from the prior source.

**Production-oriented Databricks design:** Consider Lakeflow Declarative Pipelines **AUTO CDC**, `KEYS`, `SEQUENCE BY`, `APPLY AS DELETE WHEN`, and `STORED AS SCD TYPE 2`, alongside Unity Catalog governance, pipeline monitoring, expectations, and orchestration. Verify workspace/Free Edition feature availability before running AUTO CDC examples.

## STEP 28 — Solutions Architect interview readiness (English)

**Q1. Why not use `whenMatchedUpdateAll()` for customer SCD2?**  
Because it overwrites the current customer version and loses the historical state. A staged MERGE expires the previous row and inserts a new dated version.

**Q2. How do you avoid processing replayed CDC events twice?**  
Use source event identities to deduplicate the batch, compare against already committed source-time watermarks, and make changes conditional. An enterprise design also needs durable replay/checkpoint tracking.

**Q3. What is the difference between event time and ingestion time?**  
Event time expresses when the business change occurred. Ingestion time expresses when our data platform received the event. Using arrival order as business order can corrupt historical results.

**Q4. What would you do with an UPDATE that has only one changed attribute?**  
Treat it as a PATCH, preserving attributes not explicitly supplied. Do not assume omitted values mean NULL unless the source contract says so.

**Q5. Why doesn't a payment status UPDATE increase revenue?**  
Because it modifies an existing payment attempt's lifecycle state; it does not represent a new distinct monetary movement. Charge, capture, refund, and decline must be modeled carefully.

**Q6. What happens if a late customer event arrives after a newer version has already been committed?**  
A simple current-record MERGE is insufficient to rewrite the appropriate historical interval. I would consider Lakeflow AUTO CDC with reliable `SEQUENCE BY`, or implement an explicit bitemporal replay strategy if the business requires it.

**Q7. Why copy LAB 03/05 tables instead of directly updating them?**  
For a safe, repeatable demonstration: independent targets let us rehearse CDC behaviors without destroying trusted earlier-lab checkpoints.

### Official references

- [Delta Lake MERGE](https://docs.databricks.com/aws/en/delta/merge)
- [Lakeflow AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc)
- [Change data capture in pipelines](https://docs.databricks.com/aws/en/ldp/what-is-change-data-capture)
- [Medallion architecture](https://docs.databricks.com/aws/en/lakehouse/medallion)
- [Unity Catalog volumes](https://docs.databricks.com/aws/en/volumes/)

**Next:** LAB 07 — Build Gold-layer Customer 360 and Sales Analytics from the validated current-state tables, historical customer dimension, orders, items, and payments.